# 3 - RandomForest

maintenant que tout est traité, on peut faire l'entrainement

In [1]:
import numpy as np
import pandas as pd
import sklearn as skl
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,MinMaxScaler
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report, mean_absolute_error

In [2]:
# Usine N°1
DF_U1_Train = pd.read_csv("Data/csv/train_usine1.csv")
DF_U1_Test = pd.read_csv("Data/csv/test_usine1.csv")
DF_U1_Rul = pd.read_csv("Data/csv/rul_usine1.csv")

# Usine N°2
DF_U2_Train = pd.read_csv("Data/csv/train_usine2.csv")
DF_U2_Test = pd.read_csv("Data/csv/test_usine2.csv")
DF_U2_Rul = pd.read_csv("Data/csv/rul_usine2.csv")

In [3]:
# Features utiles : (determiné dans "Anlayse"
Features = ['cycle',
            'Température sortie LPC — T24',
            'Température sortie HPC — T30',
            'Température sortie LPT — T50',
            'Pression bypass — P15',
            'Pression sortie HPC — P30',
            'Vitesse fan — Nf',
            'Vitesse cœur — Nc',
            'Rapport pression moteur — EPR',
            'Pression statique HPC — Ps30',
            'Ratio carburant/pression — phi',
            'Vitesse corrigée fan — NRf',
            'Vitesse corrigée cœur — NRc',
            'Bypass Ratio — BPR',
            'Soutirage HPC — htBleed',
            'Débit refroid. HPT — W31',
            'Débit refroid. LPT — W32']

# répartitions
X_train_U1 = DF_U1_Train[Features]
y_train_U1 = DF_U1_Train["RUL"]

In [4]:
scaler = MinMaxScaler()
X_train_U1_scaled = scaler.fit_transform(X_train_U1)       # calcule min/max sur le train

# Sur le test : appliquer les MÊMES min/max, sans recalculer
X_test_U1_scaled = scaler.transform(DF_U1_Test[Features])

# HistGradientBoostingRegressor

HistGradientBoostingRegressor est mieux que RandomForest car les arbres se corrige entre eux, même si il est plus long a entrainé. Il peut donner de meilleur résultat. Cependant il est plus sensible au bruits

In [7]:
from sklearn.ensemble import HistGradientBoostingRegressor
hgb = HistGradientBoostingRegressor(
    random_state=42,
    early_stopping=True,       # active la surveillance
    validation_fraction=0.1,   # 10% du train mis de côté pour surveiller
    n_iter_no_change=10,       # arrête si pas d'amélioration sur 10 itérations
) # Pas besoin de jobs, c'est déja optimisé
hgb.fit(X_train_U1_scaled, y_train_U1)

,"loss loss: {'squared_error', 'absolute_error', 'gamma', 'poisson', 'quantile'}, default='squared_error'The loss function to use in the boosting process. Note that the""squared error"", ""gamma"" and ""poisson"" losses actually implement""half least squares loss"", ""half gamma deviance"" and ""half poissondeviance"" to simplify the computation of the gradient. Furthermore,""gamma"" and ""poisson"" losses internally use a log-link, ""gamma""requires ``y > 0`` and ""poisson"" requires ``y >= 0``.""quantile"" uses the pinball loss... versionchanged:: 0.23 Added option 'poisson'... versionchanged:: 1.1 Added option 'quantile'... versionchanged:: 1.3 Added option 'gamma'.",'squared_error'
,"quantile quantile: float, default=NoneIf loss is ""quantile"", this parameter specifies which quantile to be estimatedand must be between 0 and 1.",None
,"learning_rate learning_rate: float, default=0.1The learning rate, also known as *shrinkage*. This is used as amultiplicative factor for the leaves values. Use ``1`` for noshrinkage.",0.1
,"max_iter max_iter: int, default=100The maximum number of iterations of the boosting process, i.e. themaximum number of trees.",100
,"max_leaf_nodes max_leaf_nodes: int or None, default=31The maximum number of leaves for each tree. Must be strictly greaterthan 1. If None, there is no maximum limit.",31
,"max_depth max_depth: int or None, default=NoneThe maximum depth of each tree. The depth of a tree is the number ofedges to go from the root to the deepest leaf.Depth isn't constrained by default.",None
,"min_samples_leaf min_samples_leaf: int, default=20The minimum number of samples per leaf. For small datasets with lessthan a few hundred samples, it is recommended to lower this valuesince only very shallow trees would be built.",20
,"l2_regularization l2_regularization: float, default=0The L2 regularization parameter penalizing leaves with small hessians.Use ``0`` for no regularization (default).",0.0
,"max_features max_features: float, default=1.0Proportion of randomly chosen features in each and every node split.This is a form of regularization, smaller values make the trees weakerlearners and might prevent overfitting.If interaction constraints from `interaction_cst` are present, only allowedfeatures are taken into account for the subsampling... versionadded:: 1.4",1.0
,"max_bins max_bins: int, default=255The maximum number of bins to use for non-missing values. Beforetraining, each feature of the input array `X` is binned intointeger-valued bins, which allows for a much faster training stage.Features with a small number of unique values may use less than``max_bins`` bins. In addition to the ``max_bins`` bins, one more binis always reserved for missing values. Must be no larger than 255.",255
,"categorical_features categorical_features: array-like of {bool, int, str} of shape (n_features) or shape (n_categorical_features,), default='from_dtype'Indicates the categorical features.- None : no feature will be considered categorical.- boolean array-like : boolean mask indicating categorical features.- integer array-like : integer indices indicating categorical features.- str array-like: names of categorical features (assuming the training data has feature names).- `""from_dtype""`: dataframe columns with dtype ""category"" are considered to be categorical features. The input must be an object exposing a ``__dataframe__`` method such as pandas or polars DataFrames to use this feature.For each categorical feature, there must be at most `max_bins` uniquecategories. Negative values for categorical features encoded as numericdtypes are treated as missing values. All categorical values areconverted to floating point numbers. This means that categorical valuesof 1.0 and 1 are treated as the same category.Read more in the :ref:`User Guide ` and:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_categorical.py`... versionadded:: 0.24.. versionchanged:: 1.2 Added support for feature names... versionchanged:: 

In [8]:
DF_result  = scaler.transform(DF_U1_Test.groupby('machine_uid').last()[Features])
# Prédictions
y_Pred_U1 = hgb.predict(DF_result)

In [9]:
y_true_U1 = DF_U1_Rul['RUL']

R2 = r2_score(y_true_U1, y_Pred_U1)
MAE = mean_absolute_error(y_true_U1, y_Pred_U1)
MSE = mean_squared_error(y_true_U1, y_Pred_U1)
RMSE = np.sqrt(MSE)

print(" ---------- Resultat ----------")
print(f"R² : {R2:.4f}")
print(f"MAE (erreur moyenne): {MAE:.2f}")
print(f"MSE (erreur au carré) : {MSE:.2f}")
print(f"RMSE (erreur écart-type) : {RMSE:.2f}")

 ---------- Resultat ----------
R² : -0.9645
MAE (erreur moyenne): 46.35
MSE (erreur au carré) : 3379.42
RMSE (erreur écart-type) : 58.13


In [10]:
# Ordre des machines dans les prédictions
print(DF_U1_Test.groupby('machine_uid').last().index[:5])

# Ordre des machines dans le RUL vrai
print(DF_U1_Rul['machine_uid'][:5])

Index(['100_1_1', '100_1_2', '10_1_1', '10_1_2', '11_1_1'], dtype='object', name='machine_uid')
0    1_1_1
1    2_1_1
2    3_1_1
3    4_1_1
4    5_1_1
Name: machine_uid, dtype: object


Les Rul est triée numériquement alors que mon test est alphabétique 

In [11]:
# L'ordre des machine_uid dans tes prédictions
ordre = DF_U1_Test.groupby('machine_uid').last().index

# Réordonner le RUL vrai dans ce même ordre
y_true = DF_U1_Rul.set_index('machine_uid').loc[ordre, 'RUL']
y_true_U1 = DF_U1_Rul['RUL']

R2 = r2_score(y_true, y_Pred_U1)
MAE = mean_absolute_error(y_true, y_Pred_U1)
MSE = mean_squared_error(y_true, y_Pred_U1)
RMSE = np.sqrt(MSE)

print(" ---------- Resultat ----------")
print(f"R² : {R2:.4f}")
print(f"MAE (erreur moyenne): {MAE:.2f}")
print(f"MSE (erreur au carré) : {MSE:.2f}")
print(f"RMSE (erreur écart-type) : {RMSE:.2f}")

 ---------- Resultat ----------
R² : 0.8169
MAE (erreur moyenne): 13.38
MSE (erreur au carré) : 314.92
RMSE (erreur écart-type) : 17.75


In [12]:
# Score PHM08 asymétrique
def phm_score_sur(rul_true, rul_pred):
    d = rul_pred - rul_true
    s = np.where(d < 0, np.exp(-d / 13) - 1, 0)
    return float(np.sum(s))

def phm_score_dang(rul_true, rul_pred):
    d = rul_pred - rul_true
    s = np.where(d > 0, np.exp(d / 10) - 1, 0)
    return float(np.sum(s))

PHMS = phm_score_sur(y_true, y_Pred_U1)
PHMD = phm_score_dang(y_true, y_Pred_U1)
print(f"PHM08 Sur : {PHMS/100}")
print(f"PHM08 Dangereux : {PHMD/100}")

PHM08 Sur : 3.8753012649750285
PHM08 Dangereux : 8.639738334214966


Le PHM08 sert a savoir si le modèle prédit en majorité + de cycle de vie ou moins.
PHMS est "Sur" car c'est lorsque le modèle prédit -
PHMD est "dangereux" car c'est lorsque le modèle prédit +, c'est a dire il surestime la durée de vie de la machine ce que l'on veut pas

Et en regardant les résultat le modèle prédit + de cycle que - de cycle

In [13]:
# Score PHM08 asymétrique
DF_PHMS =[]
def construire_table_erreurs(y_true, y_pred, machine_uid):
    d = y_pred - y_true
    df = pd.DataFrame({
        'y_true': y_true,
        'y_pred': y_pred,
        'd': d
    })
    return df

DF_PHMS = construire_table_erreurs(y_true, y_Pred_U1, y_true.index)
DF_PHMS

,y_true,y_pred,d
machine_uid,,,
100_1_1,20,13.532298,-6.467702
100_1_2,28,30.516266,2.516266
10_1_1,96,98.865444,2.865444
10_1_2,66,90.834225,24.834225
11_1_1,97,70.177385,-26.822615
...,...,...,...
98_1_2,17,19.547952,2.547952
99_1_1,117,116.202549,-0.797451
99_1_2,8,9.577222,1.577222


In [53]:
masque_risque = y_train_U1 <= 30
n_risque = masque_risque.sum()
n_total = len(y_train_U1)
proportion = n_risque / n_total
n_normaux = n_total - n_risque
print(f"Nombre de lignes a risque : {n_risque}") # donne le nombre de lignes du train où le RUL vrai est ≤ 30 
print(f"Nombre de lignes normaux : {n_normaux}") # le nombre de lignes du train "normaux"
print(f"Nombre de lignes totaux : {n_total}") # donne le nombre total de lignes du train
print(f"Pourcentage de critique dans le document : {proportion * 100}%") # dit quelle part ça représente dans tout le jeu d'entraînement.

ratio = (1 - proportion) / proportion
weights = np.where(masque_risque, 12, 1)
print(f"Facteur multiplicateur pour les 'A risque' : {ratio}")

Nombre de lignes a risque : 6200
Nombre de lignes normaux : 39151
Nombre de lignes totaux : 45351
Pourcentage de critique dans le document : 13.671142863442922%
Facteur multiplicateur pour les 'A risque' : 6.314677419354839


Ici, on voit que les zones a risques représente 13,67 % du dataset. Il faut donc ajouter un facteur de 6.3 a toute ces données afin de rééquilibrer

In [54]:
hgb2 = HistGradientBoostingRegressor(
    random_state=42,
    early_stopping=True,       # active la surveillance
    validation_fraction=0.1,   # 10% du train mis de côté pour surveiller
    n_iter_no_change=10,       # arrête si pas d'amélioration sur 10 itérations
) # Pas besoin de jobs, c'est déja optimisé
hgb2.fit(X_train_U1_scaled, y_train_U1, sample_weight=weights)

,"loss loss: {'squared_error', 'absolute_error', 'gamma', 'poisson', 'quantile'}, default='squared_error'The loss function to use in the boosting process. Note that the""squared error"", ""gamma"" and ""poisson"" losses actually implement""half least squares loss"", ""half gamma deviance"" and ""half poissondeviance"" to simplify the computation of the gradient. Furthermore,""gamma"" and ""poisson"" losses internally use a log-link, ""gamma""requires ``y > 0`` and ""poisson"" requires ``y >= 0``.""quantile"" uses the pinball loss... versionchanged:: 0.23 Added option 'poisson'... versionchanged:: 1.1 Added option 'quantile'... versionchanged:: 1.3 Added option 'gamma'.",'squared_error'
,"quantile quantile: float, default=NoneIf loss is ""quantile"", this parameter specifies which quantile to be estimatedand must be between 0 and 1.",None
,"learning_rate learning_rate: float, default=0.1The learning rate, also known as *shrinkage*. This is used as amultiplicative factor for the leaves values. Use ``1`` for noshrinkage.",0.1
,"max_iter max_iter: int, default=100The maximum number of iterations of the boosting process, i.e. themaximum number of trees.",100
,"max_leaf_nodes max_leaf_nodes: int or None, default=31The maximum number of leaves for each tree. Must be strictly greaterthan 1. If None, there is no maximum limit.",31
,"max_depth max_depth: int or None, default=NoneThe maximum depth of each tree. The depth of a tree is the number ofedges to go from the root to the deepest leaf.Depth isn't constrained by default.",None
,"min_samples_leaf min_samples_leaf: int, default=20The minimum number of samples per leaf. For small datasets with lessthan a few hundred samples, it is recommended to lower this valuesince only very shallow trees would be built.",20
,"l2_regularization l2_regularization: float, default=0The L2 regularization parameter penalizing leaves with small hessians.Use ``0`` for no regularization (default).",0.0
,"max_features max_features: float, default=1.0Proportion of randomly chosen features in each and every node split.This is a form of regularization, smaller values make the trees weakerlearners and might prevent overfitting.If interaction constraints from `interaction_cst` are present, only allowedfeatures are taken into account for the subsampling... versionadded:: 1.4",1.0
,"max_bins max_bins: int, default=255The maximum number of bins to use for non-missing values. Beforetraining, each feature of the input array `X` is binned intointeger-valued bins, which allows for a much faster training stage.Features with a small number of unique values may use less than``max_bins`` bins. In addition to the ``max_bins`` bins, one more binis always reserved for missing values. Must be no larger than 255.",255
,"categorical_features categorical_features: array-like of {bool, int, str} of shape (n_features) or shape (n_categorical_features,), default='from_dtype'Indicates the categorical features.- None : no feature will be considered categorical.- boolean array-like : boolean mask indicating categorical features.- integer array-like : integer indices indicating categorical features.- str array-like: names of categorical features (assuming the training data has feature names).- `""from_dtype""`: dataframe columns with dtype ""category"" are considered to be categorical features. The input must be an object exposing a ``__dataframe__`` method such as pandas or polars DataFrames to use this feature.For each categorical feature, there must be at most `max_bins` uniquecategories. Negative values for categorical features encoded as numericdtypes are treated as missing values. All categorical values areconverted to floating point numbers. This means that categorical valuesof 1.0 and 1 are treated as the same category.Read more in the :ref:`User Guide ` and:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_categorical.py`... versionadded:: 0.24.. versionchanged:: 1.2 Added support for feature names... versionchanged:: 

In [55]:
y_Pred_U1b = hgb2.predict(DF_result)
R2 = r2_score(y_true, y_Pred_U1b)
MAE = mean_absolute_error(y_true, y_Pred_U1b)
MSE = mean_squared_error(y_true, y_Pred_U1b)
RMSE = np.sqrt(MSE)
print(" ---------- Resultat ----------")
print(f"R² : {R2:.4f}")
print(f"MAE (erreur moyenne): {MAE:.2f}")
print(f"MSE (erreur au carré) : {MSE:.2f}")
print(f"RMSE (erreur écart-type) : {RMSE:.2f}")

 ---------- Resultat ----------
R² : 0.8264
MAE (erreur moyenne): 12.94
MSE (erreur au carré) : 298.65
RMSE (erreur écart-type) : 17.28


In [56]:
PHMS = phm_score_sur(y_true, y_Pred_U1b)
PHMD = phm_score_dang(y_true, y_Pred_U1b)
print(f"PHM08 Sur : {PHMS/100}")
print(f"PHM08 Dangereux : {PHMD/100}")

PHM08 Sur : 3.7828069988580126
PHM08 Dangereux : 7.512889246956725
